# Character topic profiles

### Pakages

In [ ]:
import pandas as pd


### Loading data

In [ ]:
# Load already-integrated Person 2 output
character_spells = pd.read_csv("character_spell_df.csv")
print(character_spells.head())

   Character ID    Character Name        Spell Name         Incantation  \
0             2       Ron Weasley   Unlocking Charm           alohomora   
1            59   Filius Flitwick  Levitation Charm  wingardium leviosa   
2             3  Hermione Granger  Levitation Charm  wingardium leviosa   
3             2       Ron Weasley  Levitation Charm  wingardium leviosa   
4             3  Hermione Granger   Unlocking Charm           alohomora   

                    Topic Label   Topic_0   Topic_1   Topic_2   Topic_3  Uses  \
0  LDA Topic 2: Revealing/Other  0.037269  0.026678  0.910639  0.025414     1   
1  LDA Topic 0: Control/Utility  0.961064  0.014122  0.011362  0.013452     1   
2  LDA Topic 0: Control/Utility  0.961064  0.014122  0.011362  0.013452     1   
3  LDA Topic 0: Control/Utility  0.961064  0.014122  0.011362  0.013452     1   
4  LDA Topic 2: Revealing/Other  0.037269  0.026678  0.910639  0.025414     1   

   Topic_0_weighted  Topic_1_weighted  Topic_2_weighted  Topic

### Create weighted probabilistic character profiles


In [ ]:
topic_cols = ["Topic_0", "Topic_1", "Topic_2", "Topic_3"]

# Weight topic probabilities by spell usage frequency
for col in topic_cols:
    character_spells[col + "_weighted"] = (
        character_spells[col] * character_spells["Uses"]
    )

# Aggregate per character
character_topic_profiles = (
    character_spells
    .groupby("Character Name")
    .apply(lambda g: pd.Series({
        col: g[col + "_weighted"].sum() / g["Uses"].sum()
        for col in topic_cols
    }))
    .reset_index()
)

# Determine dominant topic
character_topic_profiles["Dominant Topic"] = (
    character_topic_profiles[topic_cols].idxmax(axis=1)
)

# Human-readable labels
topic_label_lookup = {
    "Topic_0": "Control/Utility",
    "Topic_1": "Defensive/Healing",
    "Topic_2": "Revealing/Other",
    "Topic_3": "Offensive/Dark"
}

character_topic_profiles["Character Profile"] = (
    character_topic_profiles["Dominant Topic"].map(topic_label_lookup)
)

# Preview
print(character_topic_profiles.head(10))

        Character Name   Topic_0   Topic_1   Topic_2   Topic_3 Dominant Topic  \
0     Albus Dumbledore  0.021867  0.390812  0.197250  0.390070        Topic_1   
1                  All  0.015970  0.011431  0.961709  0.010890        Topic_2   
2  Bellatrix Lestrange  0.641332  0.326218  0.014858  0.017593        Topic_0   
3     Dolores Umbridge  0.031535  0.467160  0.479802  0.021503        Topic_2   
4         Draco Malfoy  0.961064  0.014122  0.011361  0.013452        Topic_0   
5      Filius Flitwick  0.961064  0.014122  0.011362  0.013452        Topic_0   
6         Fred Weasley  0.037270  0.915852  0.021464  0.025414        Topic_1   
7       George Weasley  0.037270  0.915852  0.021464  0.025414        Topic_1   
8               Ghosts  0.025799  0.018468  0.014858  0.940875        Topic_3   
9    Gilderoy Lockhart  0.013414  0.009602  0.007725  0.969258        Topic_3   

   Character Profile  
0  Defensive/Healing  
1    Revealing/Other  
2    Control/Utility  
3    Revealing/O

C:\Users\Baxe\AppData\Local\Temp\ipykernel_31736\1274982118.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  character_spells
